# Chapter 10 Exercise: Traceability and Sign-off

Build a real traceability graph over your own two coffee-maker requirements,
close the one real gap it finds right away (mirroring real notebook 01's own
remediation, done in the same place it is found, not later), reconstruct a
judgment ledger over three of your own already-built records, and synthesize
everything into one honestly-scoped sign-off record -- mirroring
`chapters/ch10-traceability-signoff/{01-traceability-graph,
02-judgment-synthesis,03-engineering-signoff}.ipynb` one-for-one, for the
coffee-maker domain instead of the toaster's own.

## Problem

This chapter adds exactly one new model element, in either domain, and only
because its own traceability analysis finds a real gap it is positioned to
close directly: main Chapter 10 queries `models/ch10-cumulative.sysml`
directly, which otherwise carries `models/ch08-cumulative.sysml`'s content
forward unchanged (the same design choice Chapter 9 made) -- unlike Chapter
9, main Chapter 10 still commits its own fixture file, but this exercise
track has never committed one for either chapter, so the same
blank-placeholder adaptation `exercises/ch09/exercise.ipynb` already
established applies here too. This exercise does the same, over your own
accumulated coffee-maker model from Chapters 1-8, with the same adaptations
the exercise track already needs, since exercises never commit a solution
file (no answer key):

- **No committed model fixture.** Step 1 below is a blank placeholder: paste
  your own full, completed Chapters 1-8 coffee-maker model as `source`, the
  same way `exercises/ch09/exercise.ipynb`'s own Step 1 does.
- **The remediation in Step 2 is also a blank placeholder.** Mirroring real
  notebook 01's own fix, you write your own `MassConservationReq`-style
  requirement, with no declared `subject` (SysML v2 formal/2026-03-02
  SS7.21.1's own sanctioned degenerate case) and a `require constraint`
  subsetting `deliveredMassBoundedBySupply` directly -- deliberately with NO
  `assert satisfy` line: real notebook 01's own negative control shows that
  idiom fails under `model.verify_satisfaction()`, for the same structural
  reason it would here.
- **Two ch06-stage placeholders, not one committed file.** Step 3 needs your
  own `exercises/ch06/exercise.ipynb`'s two DIFFERENT model states, because
  that exercise's own two records were genuinely hashed against two different
  points in its own narrative: `AS-C06-EX` was hashed against
  `source_with_decomp` (before `Impeller` existed -- DL-065's own ordering
  fix: the record is written before the mechanism it selects is built), while
  `AI-C06-EX` was hashed against `source_with_impeller` (after `Impeller`
  and its `rated`/`weak` candidates exist, since evaluating `brewReq(rated)`/
  `brewReq(weak)` needs them). So Step 3 asks for TWO ch06-stage
  placeholders, `ch06_source_pre_impeller` and `ch06_source_final`, each
  hashed against the record that was actually written against it -- do not
  collapse them into one shared placeholder; that would silently change
  which model state each record's own `content_hash` points at.
- **Records are reconstructed as scaffolds, not finished text.** `AS-C06-EX`,
  `AS-C08-EX` and `AI-C06-EX` are reconstructed below exactly as
  `exercises/ch06/exercise.ipynb` and `exercises/ch08/exercise.ipynb`
  themselves already declare them -- fixed fields real, free-text fields left
  as the same `"# ..."` hints those exercises already use. The two new
  records this chapter's own Steps 2 and 4 build, `AC-C10-EX` and
  `AI-C10-EX`, keep the same split: their structural and computed fields
  (`identifier`, `kind`, `model_ref`, `premises`, `assumption_refs`,
  `evidence_refs`, `disposition`, `dependency_freshness`,
  `engineering_conclusion`, `record_kind`) are filled in below from what the
  earlier steps actually establish; their argued fields (`claim`, `scope`,
  `criteria`, `rationale`, `counterevidence`, `residual_uncertainties`) are
  left as hints for your own judgment, since they depend on how you read
  your own findings, not on a fact this notebook can compute for you.

Use only constructs and operations introduced up to Chapter 10.

## Step 1: A real traceability graph over your own two requirements

For each of your model's two named requirements, this step traces: the
requirement's own declared `subject` feature (its functional intent), the
allocation that carries it forward, the realization that specializes it,
and the verification evidence that actually exists -- then searches, in the
other direction, for whether your own model's strongest formal evidence
ties back to any requirement at all.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")

# Paste your own completed Chapters 1-8 coffee-maker model here (the same
# full source your own Chapter 9 exercise already built on). There is no
# committed exercises/models/*.sysml fixture to load instead -- exercises
# never commit a solution file, so this is the same blank placeholder every
# earlier chapter's own Step 1 cell already asks you to fill.
source = """
# Your solution here
"""

model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"
assert model.find("CoffeeDemo::deliveredMassBoundedBySupply") is not None, (
    "source is missing CoffeeDemo::deliveredMassBoundedBySupply -- paste your "
    "full Chapters 1-8 model, not an earlier stage"
)


The model loads cleanly. Before tracing anything through it, the same
language-tier control every chapter carries, chosen here to match what this
chapter actually traces: an `allocate` naming a feature that was never
declared still fails to load, which matters directly for a chapter about
following allocation and realization chains -- an allocation naming a
feature that does not exist can never silently stand in for a real one.
This control does not test a different, real gap: whether an allocation
naming a feature that DOES exist, but is the wrong one (a genuine mismatch,
not an undeclared reference), would be caught the same way. That case is
untested here.

In [ ]:
# An allocate naming an undeclared feature fails to load; it can never silently
# stand in for a real allocation link this chapter's own traceability graph follows.
bad_source = """
package Bad {
    action def A;
    part def W { perform action a : A; }
    part w : W;
    allocation badAlloc allocate A::missing to w;
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok, "Expected failure for an allocation naming an undeclared feature"
print(f"Negative control ok: bad.ok={bad.ok}")


With the model loaded, the graph starts from your model's two named
requirement usages and, for each, the requirement definition's own declared
`subject` feature: not read off the source text by eye, but found the same
way any other query in this tutorial finds a real fact, by reading each
candidate feature's own `sysx:sourceText` in the API-JSON export for the
literal `subject` keyword SysML v2 itself requires there (SysML v2
formal/2026-03-02 SS8.3, RequirementDefinition) -- the same
`requirement_subject()` helper real notebook 01 builds, mirrored here since
`src/toaster/query.py` has not gained an equivalent.

In [ ]:
from toaster.query import ApiIndex, find_requirements

idx = ApiIndex(model)


def requirement_subject(idx, req_def_qn):
    """The (name, type) of req_def_qn's own declared `subject` feature, found by
    reading each of its owned ReferenceUsage features for the literal `subject`
    keyword in its own sysx:sourceText, not assumed from the requirement's name."""
    for e in idx.of_type("ReferenceUsage"):
        qn = e.get("qualifiedName", "")
        if qn.startswith(req_def_qn + "::") and e.get("sysx:sourceText", "").strip().startswith("subject "):
            return e["declaredName"], idx.type_names(qn)[0]
    return None, None


requirements = sorted(r.id for r in find_requirements(model))
for r in requirements:
    req_def = idx.type_names(r)[0]
    subject_name, subject_type = requirement_subject(idx, req_def)
    print(f"{r}: def={req_def}  subject={subject_name} : {subject_type}")


Report what your own model's `requirement_subject()` output actually
names for each requirement's own subject -- do not assume it matches
`brewReq`'s subject being a `WaterMover` and `tempCheck`'s subject being a
`BrewUnit` without confirming it against your own printed output above; the
rest of this notebook assumes those two names, but only because they are
this domain's own real, already-established construct names
(`exercises/ch02`, `exercises/ch03`, `exercises/ch06`), not because this
notebook can force your model to have them.

In [ ]:
from toaster.query import allocations_for, supertypes_transitively

brewreq_allocations = allocations_for(model, "CoffeeDemo::BrewAssembly::mover", inherit=True, index=idx)
brewreq_realizers = supertypes_transitively(model, "CoffeeDemo::Impeller")
print("brewReq's subject is allocated from:")
for a in brewreq_allocations:
    print(f"  {a}")
print(f"Impeller (the realizer chosen for WaterMover) itself specializes: {brewreq_realizers}")


`moverAllocation` allocates `applyWater.moveWater` to `mover`, declared
inside `BrewAssembly` where both resolve, the functional action Chapter 4
defined reaching the logical carrier; `Impeller` specializes `WaterMover`
(Chapter 6's own mechanism selection, `AS-C06-EX`), and `rated`/`weak` are
its two physical candidates. The same pattern, traced for `tempCheck`'s own
subject next.

In [ ]:
from toaster.query import specializes_transitively

tempcheck_allocations = allocations_for(model, "CoffeeDemo::CoffeeMaker::brewUnit", inherit=True, index=idx)
brewunit_candidates_raw = specializes_transitively(model, "CoffeeDemo::BrewUnit")
print("tempCheck's subject is allocated from:")
for a in tempcheck_allocations:
    print(f"  {a}")
print(f"specializes_transitively(BrewUnit), raw: {sorted(brewunit_candidates_raw)}")


`BrewUnit`, like `Toaster` for `timely`, has no further PHYSICAL subtype:
its own real candidates are usages typed directly by it. But the raw result
above is not filtered the same way real notebook 01's own
`specializes_transitively(model, "ToasterDemo::Toaster")` was: `Toaster` has
no subtype of its own in the toaster model, so filtering to elements owned
by the top-level package was enough there. `BrewUnit` genuinely differs: the
coffee domain's own `BrewAssembly :> BrewUnit` (Chapter 6) DOES specialize
`BrewUnit` itself (mirroring `HeatingAssembly :> HeatingSystem` exactly, one
level below the requirement's own subject type), so the raw result above
also contains `BrewAssembly` -- a `PartDefinition`, not a usage at all. A
real, confirmed difference between the two domains' exact query shape, not
a defect in the query: filtering to elements owned by the top-level package
AND typed as a `PartUsage` (not a definition) is what real notebook 01's
own owner-only filter did not need to do, because nothing specialized
`Toaster` itself there.

In [ ]:
brewunit_candidates = sorted(
    qn for qn in brewunit_candidates_raw
    if idx.by_qn[qn].get("owner", {}).get("@id") == "CoffeeDemo"
    and idx.by_qn[qn].get("@type") == "PartUsage"
)
print(f"filtered to real, top-level usage candidates: {brewunit_candidates}")


`brewAllocation` allocates `brew.applyWater` to `brewUnit`, declared inside
`CoffeeMaker` where both resolve; `BrewUnit`'s own real candidates are
`nominal` and `hot`. Both chains reach real physical candidates. What
verification evidence actually exists for each is the last, and most
consequential, link.

In [ ]:
from toaster.query import requirement_coverage

coverage = {c["requirement"]: c for c in requirement_coverage(model, idx)}
for r in requirements:
    print(coverage[r])


Report what your own `coverage` dict actually shows for `brewReq` and
`tempCheck` -- the assembly below assumes `brewReq` is covered on both sides
(`rated` satisfies it, `weak` fails it) and `tempCheck` is not (only `hot`'s
own negative claim, no positive claim on `nominal`), mirroring
`heatGenerationReq`/`timely`'s exact real-chapter shape, because that is
what `exercises/ch06/exercise.ipynb` and `exercises/ch02/exercise.ipynb` /
`exercises/ch03/exercise.ipynb` already establish for this domain -- confirm
it against your own printed `coverage` output above, not by assumption.

In [ ]:
traceability_graph = [
    {
        "requirement": "CoffeeDemo::brewReq",
        "functional_intent": "wm : WaterMover (a water mover rated for at least 0.003 kg/s)",
        "allocation": "moverAllocation: applyWater.moveWater -> mover (nested in BrewAssembly)",
        "realization": "Impeller (:> WaterMover); candidates rated, weak",
        "verification_evidence": (
            f"satisfied_by={coverage['CoffeeDemo::brewReq']['satisfied_by']}, "
            f"failed_by={coverage['CoffeeDemo::brewReq']['failed_by']} "
            "(bidirectional: a real positive claim AND a real negative claim)"
        ),
    },
    {
        "requirement": "CoffeeDemo::tempCheck",
        "functional_intent": "bu : BrewUnit (a brew unit at a temperature of at most 96C / 369.15 K)",
        "allocation": "brewAllocation: brew.applyWater -> brewUnit (nested in CoffeeMaker)",
        "realization": (
            "BrewUnit itself; candidates nominal, hot -- BrewAssembly "
            "(:> BrewUnit) also composes mover, brewReq's own subject: a "
            "real cross-link between the two requirement chains, not just "
            "two separate ones"
        ),
        "verification_evidence": (
            f"satisfied_by={coverage['CoffeeDemo::tempCheck']['satisfied_by']} (none), "
            f"failed_by={coverage['CoffeeDemo::tempCheck']['failed_by']} "
            "(one-sided: only a negative claim, plus an unbound verify objective)"
        ),
    },
]
for row in traceability_graph:
    print(row["requirement"] + ":")
    for key, value in row.items():
        if key != "requirement":
            print(f"  {key}: {value}")


`brewReq` traces all the way from a stated functional intent through a
real allocation and a real mechanism selection to genuine, opposite-polarity
verification evidence. `tempCheck` traces just as far through intent,
allocation and realization, but stops one link short: no candidate has ever
been positively checked against it. Both are real findings about your model
as it actually stands, not a contrast built to make one requirement look
better than the other.

One more question this graph can ask, in the other direction: does your
model's own strongest formal evidence, `deliveredMassBoundedBySupply`
(Chapter 8, proved by Z3 for every value its unbound features admit, not
merely evaluated at one point), tie back to any requirement at all? This
check went through several rounds before settling on two small,
explicitly-named, narrowly-scoped checks -- nothing broader is claimed:

- **Satisfy-by-subject**: is the lemma ever named as the SUBJECT of a real
  `assert satisfy <requirement> by <lemma>;` relationship -- this domain's
  own analog of the chapter's own idiom?
- **Direct reference from within a requirement's own body**: does any
  element owned (at any nesting depth) by a requirement definition or
  usage (an EXACT type match: a concern or viewpoint owner does not
  count, even though both are genuine metaclass subtypes) have its own
  `subsets`, `redefines`, `references` or `referent` field pointing at the
  lemma's id directly?

By design, this does **not** detect a connection-end (port/connector
typing); a `dependency`/`allocate`/`metadata about`-style relationship
naming the lemma and a requirement as two sibling elements where neither
owns the other; concern/viewpoint ownership; a transitive chain through an
intermediate element not itself owned by a requirement; or any other
relationship shape not named above (`toaster.query.requirement_ties`,
`decisions/next-passes.md` item 29, `decisions/log.md` DL-070/DL-071).
This mirror uses the exact same, unmodified library helper real notebook
01 calls. Run below to see the search itself.

In [ ]:
from toaster.query import requirement_ties, tied_to_any_requirement

delivered_mass_bounded = idx.by_qn["CoffeeDemo::deliveredMassBoundedBySupply"]
ties = requirement_ties(model, "CoffeeDemo::deliveredMassBoundedBySupply", idx)
tied_to_a_requirement = tied_to_any_requirement(model, "CoffeeDemo::deliveredMassBoundedBySupply", idx)

print(f"deliveredMassBoundedBySupply @type: {delivered_mass_bounded['@type']}")
print(f"Satisfy-by-subject or direct-reference-from-a-requirement-body hits: {ties}")
print(f"deliveredMassBoundedBySupply tied to any requirement (by these two narrow checks): {tied_to_a_requirement}")
assert not tied_to_a_requirement

Confirmed: under these two narrow checks' own stated scope, your model's
`deliveredMassBoundedBySupply` is tied to no requirement -- no `assert
satisfy` names it as a `by`-subject, and no element owned by a
requirement's own body subsets, redefines, references, or bare-names it
directly. This is not a claim that no tie could exist by any conceivable
mechanism -- only that these two specific, narrow, well-understood checks
find none (an honest, named scope limit, not a hidden one).

What IS a real, honest pair of open questions about your model as it
stands: `BrewUnit::brewTemp` remains unbound for `nominal`, and
`deliveredMassBoundedBySupply` -- this domain's own strongest proof, built
with Z3 -- ties to no requirement under this narrow check's own scope.
`BrewUnit::brewTemp` stays open (a non-goal of this step, and not something
a traceability graph alone can close); the `deliveredMassBoundedBySupply`
gap is closed directly below, precisely because finding it is what this
step's own analysis did -- not papered over by forcing a connection your
model does not actually make.

Continue to Step 2 below, which closes this gap directly.

## Step 2: close the gap Step 1 found

Frame this exactly as real Chapter 10 now does: your own model's strongest
proof, `deliveredMassBoundedBySupply`, was genuinely untied from any
requirement under Step 1's own two narrow checks -- close that gap the same
way, rather than leaving it stated. Mirror real notebook 01's own
remediation, in your own domain: a new requirement (e.g.
`MassConservationReq`) restating exactly the property
`deliveredMassBoundedBySupply` proves, with **no declared `subject`** (its
own required constraint never references a subject at all, so declaring one
would commit to an arbitrary, unused type; it inherits `RequirementCheck`'s
own default, `subject subj : Anything[1]` -- SysML v2 formal/2026-03-02
SS7.21.1, the honest reflection of that). Two different earlier drafts of
the real chapter's own construct hit two different problems here: one typed
the subject `HeatGenerator` but never added an `assert satisfy` line at
all, so its own defect (a declared, unused subject) was found by direct
spec reading, not by any tool diagnostic; a separate, independently-built
draft paired a typed subject with a real `assert satisfy` line, which the
real OMG pilot did flag directly, fixed by dropping the subject before this
reconciliation began. Do not repeat either mistake here: no declared
subject, and (see below) no `assert satisfy` line either. A `require
constraint` subsetting `deliveredMassBoundedBySupply` directly from within
the requirement's own body (Check B's own idiom) is the tie. Deliberately
absent: an `assert satisfy` line (Check A's own idiom). Real notebook 01's
own negative control shows directly why: attempting
`assert satisfy energyConservationReq by deliveredEnergyBoundedBySupply;`
makes `model.verify_satisfaction()` error identically regardless of its own
binding, because that requirement's own required constraint never
references its subject at all -- the identical structural reason applies
here, to your own `MassConservationReq` and `deliveredMassBoundedBySupply`.
As with every other step in this exercise, write your own SysML text; this
notebook does not write it for you.

In [ ]:
# Your solution here: mirror real Chapter 10's own EnergyConservationReq for
# your own coffee-maker domain. Write a requirement def (e.g.
# MassConservationReq) with NO declared subject (inheriting RequirementCheck's
# own default, Anything -- do NOT type it by WaterMover; that is the exact
# subject-type mistake the real chapter's own earlier draft made and had to
# revert) and a require constraint subsetting deliveredMassBoundedBySupply
# directly from within the requirement's own body (Check B's own idiom) --
# and deliberately NO assert satisfy line (Check A's own idiom is not used
# here; see the markdown above for why). Insert it inside your own model's
# own package, right before its closing brace, the same "the increment lands
# inside the package" idiom real notebook 01's own committed fixture uses.
mass_conservation_req_increment = """
# Your solution here
"""

remediated_source = source.rstrip()
assert remediated_source.endswith("}"), "source must end with the package's closing brace"
remediated_source = remediated_source[:-1] + mass_conservation_req_increment + "\n}\n"

remediated_model = conn.load_from_content(remediated_source, strict=False)
assert remediated_model.ok, f"Model failed: {format_diagnostics(remediated_model.diagnostics)}"


Re-run Step 1's own check against your remediated model, not a new one:
confirm it flips from `False` to `True`, attributed to your own new
requirement's DEFINITION (not its usage) -- the same way real notebook 01's
own Check-B hit is attributed to `EnergyConservationReq`, not
`energyConservationReq` -- not a forced or renamed connection to something
already there.

In [ ]:
remediated_idx = ApiIndex(remediated_model)
mass_ties_after = requirement_ties(remediated_model, "CoffeeDemo::deliveredMassBoundedBySupply", remediated_idx)
mass_tied_after = tied_to_any_requirement(remediated_model, "CoffeeDemo::deliveredMassBoundedBySupply", remediated_idx)

print(f"Satisfy-by-subject or direct-reference-from-a-requirement-body hits, after your own remediation: {mass_ties_after}")
print(f"deliveredMassBoundedBySupply tied to any requirement, after your own remediation: {mass_tied_after}")
assert mass_tied_after, (
    "Your own remediation should flip this to True -- add the require-constraint "
    "subsetting tie (Check B) from within your own new requirement's body, with "
    "no assert satisfy line"
)


In [ ]:
# From here on, `model`/`idx` refer to the remediated (now genuinely tied)
# model -- `source` itself is left untouched, since AS-C08-EX (Step 3 below)
# is hashed against the Chapters 1-8 state it was actually written against,
# not this one.
model = remediated_model
idx = remediated_idx


Your own model's strongest proof was genuinely untied; you just closed that
gap the same way real Chapter 10 does -- structurally, by subsetting, not by
`assert satisfy`. This is the construct-and-analyze loop the whole tutorial
teaches, run once more, in your own domain: build, query, find a gap,
remediate, re-query, confirm.

Real Chapter 10 does not stop at closing the tie: it also builds a judgment
record, `AC-C10`, that honestly frames what the tie does and does not
establish -- including why no `assert satisfy` claim is given (real notebook
01's own negative control demonstrates this directly, rather than only
arguing it in a doc comment). Build your own version here, `AC-C10-EX`, the
same blank-scaffold way every other record in this exercise track is built:
structural and computed fields filled in from what you have shown above,
argued fields left as hints for your own judgment.

In [ ]:
ac_c10_ex_premises = [
    "SysML v2.0 formal/2026-03-02 SS7.21.2 explicitly supports "
    "reference-subsetting an existing constraint as a requirement's own "
    "required constraint (`require <name>;` / `require constraint c :> "
    "<name>;`) -- the spec's own intended idiom for reusing an "
    "already-declared constraint, confirmed by reading the spec directly.",
    "SS7.21.1: 'A requirement usage can only be satisfied by an entity that "
    "conforms to the definition of its subject.' The base `requirement def "
    "RequirementCheck` (Systems Library/Requirements.sysml) declares "
    "`subject subj : Anything[1];`; your own new requirement leaves its own "
    "subject undeclared, inheriting that default, the same fix real "
    "notebook 01 confirms directly against the base library.",
    f"Satisfy-by-subject or direct-reference-from-a-requirement-body hits, "
    f"after your own remediation: {mass_ties_after} (tied={mass_tied_after}).",
    "# State plainly, mirroring real AC-C10: did your own new requirement's "
    "need predate or postdate the gap Step 1 found? Say which, honestly.",
]
print(ac_c10_ex_premises)


In [ ]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

ac_c10_ex = ReviewRecord(
    identifier="AC-C10-EX",
    kind="asserted_context",
    claim="# State plainly, mirroring real AC-C10: is your own new requirement's "
          "tie to deliveredMassBoundedBySupply spec-legitimate, is its own general "
          "motivation genuine and independent of any one proof, and did its own "
          "need predate or postdate the gap Step 1 found? Say which, honestly.",
    subject_ref="ToasterDemo::EnergyConservationReq",
    model_ref="# Your own new requirement usage's qualified name, e.g. "
               "CoffeeDemo::massConservationReq",
    content_hash=hash_content(remediated_source),
    scope="# EnergyConservationReq/energyConservationReq's own equivalent scope, "
          "restated for your own new requirement: not a claim about "
          "deliveredMassBoundedBySupply's own proof (AS-C08-EX already states "
          "that proof's real limits) or about any other requirement in your model.",
    criteria="# Mirror real AC-C10's own three criteria: (a) does the construct "
             "conform to the requirement's own declared subject type? (b) is the "
             "requirement's own need independently motivated, or does its content "
             "trace back only to the evidence it was built to accommodate? (c) does "
             "requirement_coverage()'s own covered=False result for it honestly "
             "reflect a deliberate design choice, rather than reading as a second "
             "version of tempCheck's own real gap?",
    premises=ac_c10_ex_premises,
    assumption_refs=[
        "AS-C08-EX's own residual_uncertainties (your own Chapter 8 exercise): "
        "deliveredMassBoundedBySupply is a hand-restated companion lemma, not a "
        "solver-checked reference to WaterMover's own real elements; this record "
        "does not strengthen that proof, only frames how its tie to a requirement "
        "should be read."
    ],
    evidence_refs=[
        "Systems Library/Requirements.sysml's own default subject, confirmed in "
        "real notebook 01 and cited above.",
        f"requirement_ties(remediated_model, LEMMA, idx) after remediation: "
        f"{mass_ties_after}",
        "# If you build it: model.verify_satisfaction() against an "
        "assert-satisfy attempt, the same negative control real notebook 01 "
        "runs, confirming it fails here too.",
    ],
    rationale="# Why the tie is real (Check B alone) and why no assert-satisfy "
              "claim is given (mirror real AC-C10's own three-part rationale)",
    counterevidence="# Mirror real AC-C10's own counterevidence: is a requirement "
                     "invented after the fact to absorb existing evidence "
                     "circular? What keeps it from being fully circular, and what "
                     "a reader could still reasonably disagree with",
    residual_uncertainties="# What is genuinely open: mirror real AC-C10's own "
                            "two residuals, restated for your own domain",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(ac_c10_ex)
print(f"AC-C10-EX validation errors (structural fields only -- fill in the argued fields above before treating this as done): {errors}")


`validate_record` reports no errors on the structural fields; fill in the
argued fields (`claim`, `scope`, `criteria`, `rationale`, `counterevidence`,
`residual_uncertainties`) with your own reasoning before treating this
record as done. Continue to Step 3 below, which reconstructs a judgment
ledger over three of your own already-built records.

## Step 3: A judgment ledger, over three of your own already-built records

This step reconstructs a judgment ledger over three real `ReviewRecord`s
you already built: `AS-C06-EX` and `AI-C06-EX`
(`exercises/ch06/exercise.ipynb`, a mechanism-selection judgment and this
domain's own stopping judgment) and `AS-C08-EX`
(`exercises/ch08/exercise.ipynb`, grounded in your own Chapter 8 Z3 proof).
This tutorial has no central registry of every `ReviewRecord` it has ever
built, the same finding Chapter 9 made when it searched for one: each
chapter's exercise constructs its own records as local Python objects, with
nothing shared beyond the file each one lives in. So this step does not
scan "every record you have ever built"; it reconstructs these three
verbatim, field for field, exactly as their own source exercises already
declare them.

In [ ]:
from toaster.evidence import ReviewRecord, hash_content, validate_record

incomplete_inference = ReviewRecord(
    identifier="AI-BAD-EX",
    kind="asserted_inference",
    claim="The judgment ledger is complete.",
    subject_ref="ToasterDemo::HeatingAssembly::heatGen",
    model_ref="CoffeeDemo",
    content_hash=hash_content(source),
    scope="CoffeeDemo",
    criteria="Every real ReviewRecord you have built is included.",
    premises=[],   # intentionally empty
    rationale="This exercise built a ledger.",
    counterevidence="No central registry exists to check completeness against.",
    record_kind="worked_example",
)
errors = validate_record(incomplete_inference)
assert len(errors) > 0, "Expected validation to fail on empty premises"
print(f"Negative control ok: errors={errors}")


Before reconstructing the three records, two ch06-stage placeholders:
`AS-C06-EX` was hashed, in your own `exercises/ch06/exercise.ipynb`, against
`source_with_decomp` (the model state at Step 2 there, before `Impeller`
existed -- DL-065's own ordering fix: the record is written before the
mechanism it selects is built). `AI-C06-EX` was hashed there against
`source_with_impeller` (the model state at Step 4/5, after `Impeller` and
its `rated`/`weak` candidates exist, since evaluating `brewReq(rated)`/
`brewReq(weak)` needs them). These are two genuinely different model
states, so this step asks for two separate placeholders, not one shared
one -- collapsing them would silently move one record's own `content_hash`
onto a state it was never actually written against.

In [ ]:
# Paste your own exercises/ch06/exercise.ipynb Step 2 source_with_decomp here
# -- the model state AS-C06-EX was actually hashed against, before Impeller
# existed.
ch06_source_pre_impeller = """
# Your solution here (paste your own exercises/ch06 Step 2 source_with_decomp --
# the model state AS-C06-EX was actually hashed against, before Impeller)
"""


In [ ]:
# Confirm ch06_source_pre_impeller is genuinely the pre-Impeller ch06 STEP 2
# stage (source_with_decomp), not left blank, not ch06 STEP 1 (source --
# WaterMover alone already exists there, before BrewAssembly/brewReq, so a
# WaterMover-only check would silently pass on Step 1 too), and not a
# later-stage model: a check that only asserted "the string changed" would
# pass even on an unfilled or wrong placeholder, the exact failure mode this
# exercise track already caught and fixed once (exercises/ch09/exercise.ipynb's
# own ch06_source guard).
ch06_pre_model = conn.load_from_content(ch06_source_pre_impeller, strict=False)
assert ch06_pre_model.ok, f"ch06_source_pre_impeller failed to load: {format_diagnostics(ch06_pre_model.diagnostics)}"
assert ch06_pre_model.find("CoffeeDemo::WaterMover") is not None, (
    "ch06_source_pre_impeller must declare WaterMover -- paste your own ch06 "
    "Step 2 model, not a blank string"
)
assert (
    ch06_pre_model.find("CoffeeDemo::BrewAssembly") is not None
    and ch06_pre_model.find("CoffeeDemo::brewReq") is not None
), (
    "ch06_source_pre_impeller declares WaterMover but not BrewAssembly/brewReq "
    "-- this is ch06 STEP 1 (source), before BrewAssembly and brewReq exist, "
    "not Step 2 (source_with_decomp); paste your own ch06 Step 2 model instead"
)
assert ch06_pre_model.find("CoffeeDemo::Impeller") is None, (
    "ch06_source_pre_impeller already declares Impeller -- paste your ch06 Step 2 "
    "source_with_decomp (before Impeller), not a later-stage model"
)
print("ch06_source_pre_impeller confirmed: loads cleanly, declares WaterMover/BrewAssembly/brewReq, predates Impeller.")


In [ ]:
# Paste your own exercises/ch06/exercise.ipynb Step 5 source_with_impeller
# here -- the model state AI-C06-EX was actually hashed against, after
# Impeller and its rated/weak candidates exist but before Chapter 7's own
# transferEfficiency addition.
ch06_source_final = """
# Your solution here (paste your own exercises/ch06 Step 5 source_with_impeller --
# the model state AI-C06-EX was actually hashed against, after Impeller/rated/weak,
# before Chapter 7)
"""


In [ ]:
# Confirm ch06_source_final is genuinely the post-Impeller ch06 stage, not
# left blank, not the pre-Impeller stage, and not a later chapter's own
# growth: the same real, falsifiable guard as the cell above, adapted to
# this placeholder's own two boundary conditions.
ch06_final_model = conn.load_from_content(ch06_source_final, strict=False)
assert ch06_final_model.ok, f"ch06_source_final failed to load: {format_diagnostics(ch06_final_model.diagnostics)}"
assert ch06_final_model.find("CoffeeDemo::Impeller") is not None, (
    "ch06_source_final must declare Impeller -- paste your own ch06 Step 5 "
    "source_with_impeller, not the pre-Impeller state"
)
assert ch06_final_model.eval("CoffeeDemo::brewReq(CoffeeDemo::rated)") is True, (
    "ch06_source_final's own rated candidate must satisfy brewReq"
)
assert ch06_final_model.find("CoffeeDemo::WaterMover::transferEfficiency") is None, (
    "ch06_source_final already declares WaterMover::transferEfficiency (a Chapter 7 "
    "addition) -- paste your ch06 Step 5 source_with_impeller, not your current "
    "(ch08-stage) model"
)
print("ch06_source_final confirmed: loads cleanly, declares Impeller, brewReq(rated) holds, predates transferEfficiency.")

`AS-C06-EX`, reconstructed below exactly as `exercises/ch06/exercise.ipynb`
declares it: its own `premises` are real, filled text (not a hint); its own
free-text fields stay the same `"# ..."` hints that exercise already uses --
fill in your own argued prose there, not a restated answer copied from this
one's prompt.

In [ ]:
brew_start = ch06_pre_model.find("CoffeeDemo::BrewStart")
brew_cancel = ch06_pre_model.find("CoffeeDemo::BrewCancel")
print(f"BrewStart: {brew_start.kind}, BrewCancel: {brew_cancel.kind}")

as_c06_ex = ReviewRecord(
    identifier="AS-C06-EX",
    kind="asserted_solution",
    claim="# Which mechanism WaterMover commits to, and which plausible alternative it is selected over",
    subject_ref="ToasterDemo::ResistanceCoil",
    model_ref="CoffeeDemo::WaterMover",
    content_hash=hash_content(ch06_source_pre_impeller),
    scope="CoffeeDemo::WaterMover and its realizations",
    criteria="# What the chosen mechanism must be able to do: respond to a discrete control signal, expose a rating brewReq can check",
    premises=[
        "CoffeeDemo declares BrewStart and BrewCancel (Chapter 4), "
        "discrete signals marking the start and cancellation of a "
        "brewing cycle, confirmed by model.find() above.",
        "Domain premise, not derived from the model: a motor-driven "
        "impeller responds to being switched on and off directly, "
        "while a gravity-fed drip mechanism has no actuation point of "
        "its own and needs a separate valve or timing mechanism to "
        "start or stop the flow on command. Neither WaterMover nor any "
        "of its realizations is yet connected to a control signal in "
        "this model; this premise is about the physical mechanisms "
        "themselves, not about what the model currently wires "
        "together.",
    ],
    assumption_refs=["# What WaterMover itself does NOT already commit to -- this record is what actually chooses the mechanism"],
    evidence_refs=[
        f"model.find('CoffeeDemo::BrewStart') resolves to a real {brew_start.kind}, confirmed above.",
        f"model.find('CoffeeDemo::BrewCancel') resolves to a real {brew_cancel.kind}, confirmed above.",
    ],
    rationale="# Why this mechanism is preferred, argued from how the two mechanisms work -- not from anything the model currently wires together",
    counterevidence="# What this selection does not rule out, and what stays unmodeled (mirror AS-C06's own challenge about Joule heating)",
    residual_uncertainties="# What would let this selection be revisited against a real trade study",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(as_c06_ex)
print(f"AS-C06-EX validation errors: {errors}")


`AS-C08-EX`, reconstructed the same way, verbatim, from
`exercises/ch08/exercise.ipynb`: the record grounded in your own Chapter 8
Z3 proof of `deliveredMassBoundedBySupply`, the same one Step 1 above found
tied to no requirement and Step 2 then tied, by subsetting, to your own new
requirement (see Step 1's own note, Step 2, and `decisions/next-passes.md`
item 29). Its `content_hash` is computed against `source` (Step 1's own
model, before Step 2's own remediation), the model it was actually written
against.

In [ ]:
as_c08_ex = ReviewRecord(
    identifier="AS-C08-EX",
    kind="asserted_solution",
    claim=(
        "# deliveredMassBoundedBySupply, a hand-restated real-arithmetic "
        "lemma of the same shape as WaterMover's transferEfficiencyBounded "
        "constraint and deliveredMass's own definition, holds for every "
        "value of transferEfficiency in [0,1] and every non-negative "
        "throughput and duration a companion restatement admits"
    ),
    subject_ref="ToasterDemo::deliveredEnergyBoundedBySupply",
    model_ref="CoffeeDemo::deliveredMassBoundedBySupply",
    content_hash=hash_content(source),
    scope=(
        "# The lemma governs the companion restatement's own "
        "moverCheck.transferEfficiency, moverCheck.throughput and "
        "moverCheckDuration features; state plainly that it is not a "
        "solver-checked reference to WaterMover's own "
        "transferEfficiencyBounded or deliveredMass (DEFERRED.md D-030, D-031)"
    ),
    criteria=(
        "# What verify_holds() must report for this to count as proved: a "
        "single satisfied verdict, with z3 named in the reason text, not "
        "propagation alone"
    ),
    premises=[],
    assumption_refs=[],
    evidence_refs=[
        "# paste your own verify_holds() result string from "
        "exercises/ch08/exercise.ipynb Step 2 here (the pos_verdict printed there)"
    ],
    rationale=(
        "# Why this is a materially different kind of evidence from "
        "verify_satisfaction()'s point evaluation, and why it is a "
        "narrower claim than proving WaterMover's own conservation property"
    ),
    counterevidence=(
        "# State plainly: this proof is NOT a solver-checked reference to "
        "WaterMover's own transferEfficiencyBounded or deliveredMass "
        "(DEFERRED.md D-030, D-031) -- confirmed directly in your own "
        "Chapter 8 exercise: a sibling constraint (reliesOnSibling) that "
        "depends entirely on transferEfficiencyBounded's own stated bound, "
        "without restating it, is left undecided with a genuine witness "
        "violating that very bound, because this toolchain's Z3 backend "
        "never composes two separately declared assert constraints. Also "
        "cite D-029: the companion file is used instead of the committed "
        "model directly because toaster.modelcheck's own text parser "
        "cannot yet read a verdict line for a constraint that is also the "
        "subject of an assert satisfy declaration."
    ),
    residual_uncertainties=(
        "# What this proof does not establish: whether transferEfficiency, "
        "throughput and duration ever take values outside the bound in a "
        "real candidate; whether the restated copy still matches "
        "WaterMover's own construct if either is edited later; whether any "
        "physical impeller instance's transferEfficiency/throughput values, "
        "beyond rated/weak, have been checked against this property -- "
        "WaterMover itself remains an abstract carrier with no concrete "
        "realization of its own"
    ),
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="supported",
    record_kind="worked_example",
)

errors = validate_record(as_c08_ex)
print(f"AS-C08-EX validation errors: {errors}")


`AI-C06-EX`, the third record, is genuinely different in kind: an
`asserted_inference` about this domain's own level-2 water-moving branch's
stopping judgment, rebuilt verbatim from `exercises/ch06/exercise.ipynb`
Step 5. Its `premises` field names four earlier judgment identifiers
directly (`AC-C06-EX`, `AS-C06-EX`, `AS-C03-EX`, `AI-C04-EX`), the clearest
example in this domain of Hawkins' own "child claims supporting a parent"
idea. Its `evidence_refs` cites the same real analysis results that
exercise gathered, recomputed below against a freshly loaded
`ch06_source_final` rather than restated by hand. Its `content_hash` is
also computed against that same file, since `AI-C06-EX` was written against
this domain's own post-`Impeller` Chapter 6 model, not the pre-`Impeller`
state `AS-C06-EX` used.

In [ ]:
from toaster.query import find_allocations, perform_relationships

ch06_final_model = conn.load_from_content(ch06_source_final, strict=False)
assert ch06_final_model.ok

performs = perform_relationships(ch06_final_model)
allocations = find_allocations(ch06_final_model)
rated_holds = ch06_final_model.eval("CoffeeDemo::brewReq(CoffeeDemo::rated)")
weak_holds = ch06_final_model.eval("CoffeeDemo::brewReq(CoffeeDemo::weak)")
print(f"Perform relationships: {performs}")
print(f"Allocations: {allocations}")
print(f"brewReq(rated) = {rated_holds}, brewReq(weak) = {weak_holds}")


Report what your own `performs`/`allocations` output actually shows --
note one already-flagged, real divergence from the toaster's own exact
shape (`decisions/log.md` DL-065): `BrewAssembly` declares its own `perform
action applyWater : ApplyWater;` directly, rather than inheriting it from
`BrewUnit` the way `HeatingAssembly` inherits `perform action applyHeat`
from `HeatingSystem` -- because `exercises/ch05/exercise.ipynb`'s own
`BrewUnit` was never given that perform the way main Chapter 5 gives
`HeatingSystem` one. So `perform_relationships` here shows `BrewAssembly`
performing `ApplyWater`, not `BrewUnit` -- a known, already-recorded
divergence, not a new defect.

In [ ]:
ai_c06_ex = ReviewRecord(
    identifier="AI-C06-EX",
    kind="asserted_inference",
    claim="# State plainly what this branch's structural decomposition establishes (MoveWater performed and allocated, brewReq evaluated on two real candidates) and what it does not (BrewUnit's decomposition is NOT complete)",
    subject_ref="ToasterDemo::HeatingAssembly::heatGen",
    model_ref="CoffeeDemo::BrewAssembly::mover",
    content_hash=hash_content(ch06_source_final),
    scope="CoffeeDemo::BrewAssembly::mover and its realizations",
    criteria="# Per the recursion's own stopping rule: performs / connects / verified, condition by condition, for THIS level only -- mirror AI-C06's own criteria exactly",
    premises=["AC-C06-EX", "AS-C06-EX", "AS-C03-EX", "AI-C04-EX"],
    assumption_refs=["# What is printed/loaded above that this record takes as given, without itself re-arguing it"],
    evidence_refs=[
        f"perform_relationships(model): {performs}",
        f"find_allocations(model): {allocations}",
        f"brewReq(rated) = {rated_holds}, brewReq(weak) = {weak_holds}, evaluated directly against the loaded model above.",
    ],
    rationale="# Performs / Connects / Verified, condition by condition -- and make sure each of the four premises actually does work in this paragraph, not just sits in the list",
    counterevidence="# What this branch does NOT show: waterIn unwired, BrewAssembly not composed into any CoffeeMaker candidate, ApplyWater's other flows not decomposed -- mirror AI-C06's own challenge",
    residual_uncertainties="# What is open for whichever exercise takes it up next",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(ai_c06_ex)
print(f"AI-C06-EX validation errors: {errors}")


All three validate cleanly. The ledger below reports each record's kind
(Hawkins' three sites), its `disposition` and `record_kind` (SA-7: always
`pending`, always `worked_example`, never the forbidden alternatives), and
its `engineering_conclusion`, next to what its own `residual_uncertainties`
says is NOT yet resolved.

In [ ]:
ledger = {"AS-C06-EX": as_c06_ex, "AS-C08-EX": as_c08_ex, "AI-C06-EX": ai_c06_ex}

for identifier, record in ledger.items():
    print(f"{identifier}:")
    print(f"  kind: {record.kind}")
    print(f"  disposition: {record.disposition}")
    print(f"  record_kind: {record.record_kind}")
    print(f"  engineering_conclusion: {record.engineering_conclusion}")
    print(f"  residual_uncertainties: {record.residual_uncertainties}")
    print()

assert all(r.disposition == "pending" for r in ledger.values())
assert all(r.record_kind == "worked_example" for r in ledger.values())


Three real records, three different points of confidence -- read your
own filled-in free-text fields the way `exercises/ch09/exercise.ipynb`'s own
sufficiency step reads them: is each record's `counterevidence` a specific
competing design or a specific unmodeled relation, or does it read like a
hedge that names nothing checkable? Continue to Step 4 below, which
synthesizes this ledger together with Step 1's own traceability graph and
Step 2's own remediation.

## Step 4: A synthesis record, and what it is not

State the distinction this step rests on before building anything: a
completed traceability graph and a judgment ledger are not sign-off itself.
Sign-off is a human, accountable act; a design's own engineer decides,
given everything the graph and the ledger show, whether to proceed. This
step can show the inputs to that decision, honestly and within its own
stated scope, but it cannot make the decision for you, and it does not try
to. The record built below stays `disposition = "pending"`, the same as
every other record this tutorial has ever built (SA-7): this exercise does
not get to be the exception.

Before assembling anything, a real negative control fitting this step's
own new record: does `validate_record()` reject a draft of it whose own
`counterevidence` is empty? `counterevidence` is exactly the field
AGENTS.md 1.6 calls load-bearing, and this check is real and mechanized.

In [ ]:
draft_missing_counterevidence = ReviewRecord(
    identifier="AI-C10-DRAFT-EX",
    kind="asserted_inference",
    claim="A draft of the synthesis record built below, missing its own counterevidence.",
    model_ref="CoffeeDemo",
    content_hash=hash_content(source),
    scope="CoffeeDemo",
    criteria="The same criteria the real record below states.",
    premises=["a premise"],
    rationale="Because the graph and the ledger say so.",
    counterevidence="",   # intentionally empty
    disposition="pending",
    record_kind="worked_example",
)
errors = validate_record(draft_missing_counterevidence)
assert len(errors) > 0, "Expected validation to fail on empty counterevidence"
print(f"Negative control ok: errors={errors}")


`validate_record()` correctly rejects this: a real, mechanized floor
actually catching something. A different rule this tutorial follows just as
strictly has no such mechanized floor: AGENTS.md 1.6 forbids ever recording
`disposition="accepted"` (SA-7), but reading `src/toaster/evidence.py`'s own
`validate_record()` shows what it actually checks: `identifier`, `claim`,
`rationale` and `counterevidence` non-empty, `record_kind != "actual_review"`,
and at least one premise for an `asserted_inference`, never the
`disposition` value itself. That gap is real and worth naming plainly, but
it is not demonstrated here by constructing the forbidden value: AGENTS.md's
own rule against `disposition="accepted"` has no negative-control exception,
so this point is made in prose only, never in code. Every record this
notebook builds keeps `disposition="pending"`, checked explicitly below, not
merely asserted.

With that settled, the two real inputs this synthesis draws on: Step 1's
own coverage-and-tie finding, recomputed directly below against the
loaded model (this notebook's own cells all share one running kernel, but
recomputing rather than only reusing the printed values makes the synthesis
independently checkable), and a short summary of Step 3's own three-record
ledger, cited by identifier and conclusion rather than rebuilt a third
time.

In [ ]:
delivered_mass_bounded_s4 = idx.by_qn["CoffeeDemo::deliveredMassBoundedBySupply"]
tied_to_a_requirement_s4 = tied_to_any_requirement(model, "CoffeeDemo::deliveredMassBoundedBySupply", idx)

print(f"brewReq: {coverage['CoffeeDemo::brewReq']}")
print(f"tempCheck: {coverage['CoffeeDemo::tempCheck']}")
print(f"deliveredMassBoundedBySupply tied to any requirement: {tied_to_a_requirement_s4}")

assert coverage["CoffeeDemo::brewReq"]["covered"] is True
assert coverage["CoffeeDemo::tempCheck"]["covered"] is False
assert tied_to_a_requirement_s4

In [ ]:
ledger_summary = {
    "AS-C06-EX": {"kind": as_c06_ex.kind, "engineering_conclusion": as_c06_ex.engineering_conclusion},
    "AS-C08-EX": {"kind": as_c08_ex.kind, "engineering_conclusion": as_c08_ex.engineering_conclusion},
    "AI-C06-EX": {"kind": ai_c06_ex.kind, "engineering_conclusion": ai_c06_ex.engineering_conclusion},
}
for identifier, summary in ledger_summary.items():
    print(f"{identifier}: {summary}")


Both inputs are in hand. The rest of this step builds one new record
synthesizing them, following the judgment-record construction zone
(`toaster-review-protocol`): name each group of fields, narrate what it is
for, then assemble. `kind="asserted_inference"` is the deliberate choice
here, not `asserted_solution`: this record's own premises are literally
child claims (the ledger's three records, plus `AC-C10-EX`, plus Step 1's
own coverage and tie findings) supporting a parent synthesis, Hawkins' own
"child claims supporting a parent" idea (SS3.1), not new evidence of its
own.

The fields below split the same way this exercise track's other
reconstructed records do, but not identically: `premises_s4` alone is
computed live, string-built directly from `coverage` and the three ledger
records' own attributes below (not a judgment call) -- with one named
exception: the clause about `BrewTempTest`'s own unbound verify objective
is manually stated, not computed, the same way real
`03-engineering-signoff.ipynb`'s own identical clause is (recomputing it
would need the same `sysx:declaredKeyword == "verify"` pattern
`exercises/ch09/exercise.ipynb` already uses, not attempted here).
`assumption_refs_s4` and `evidence_refs_s4` are fixed-but-real strings, not
computed from a live query, though what they describe is genuinely true of
this notebook's own prior cells. `claim`, `scope`, `criteria`, `rationale`,
`counterevidence` and `residual_uncertainties` are left as hints for your
own argued reading of what your own graph and ledger show -- fill them in
with your own reasoning, not a restated answer copied from this prompt.

In [ ]:
premises_s4 = [
    f"Step 1's own traceability graph: brewReq covered={coverage['CoffeeDemo::brewReq']['covered']} "
    f"(satisfied_by={coverage['CoffeeDemo::brewReq']['satisfied_by']}, "
    f"failed_by={coverage['CoffeeDemo::brewReq']['failed_by']}); "
    f"tempCheck covered={coverage['CoffeeDemo::tempCheck']['covered']} "
    f"(satisfied_by={coverage['CoffeeDemo::tempCheck']['satisfied_by']}, "
    f"failed_by={coverage['CoffeeDemo::tempCheck']['failed_by']}, plus BrewTempTest's "
    f"own unbound verify objective -- manually stated, not computed); "
    f"deliveredMassBoundedBySupply, found tied to no requirement under two narrow, "
    f"explicitly-named checks (satisfy-by-subject; a direct "
    f"subsets/redefines/references/referent reference from within a "
    f"requirement's own body, exact-type owner only), is now tied "
    f"(tied={tied_to_a_requirement_s4}) to your own new requirement added in Step 2 "
    f"-- not a claim that no other tie could exist by any conceivable mechanism, "
    f"only that this one, real, is closed "
    f"(decisions/next-passes.md item 29, decisions/log.md DL-070/DL-071/DL-072).",
    f"AS-C06-EX (asserted_solution, Chapter 6): a mechanism selection for "
    f"WaterMover; engineering_conclusion={as_c06_ex.engineering_conclusion!r}.",
    f"AS-C08-EX (asserted_solution, Chapter 8): deliveredMassBoundedBySupply "
    f"proved by Z3; engineering_conclusion={as_c08_ex.engineering_conclusion!r}.",
    f"AI-C06-EX (asserted_inference, Chapter 6): the level-2 water-moving "
    f"branch's own stopping judgment; engineering_conclusion={ai_c06_ex.engineering_conclusion!r}.",
    "AC-C10-EX (asserted_context, Step 2 above): frames your own new "
    "requirement's tie honestly -- the subsetting mechanism is spec-legitimate "
    "and its own general motivation is genuine, but its own need was "
    "identified in response to Step 1's own traceability gap; the tie rests "
    "on structural detection alone (Check B), deliberately not reinforced by "
    "any assert-satisfy claim.",
]
assumption_refs_s4 = [
    "CoffeeDemo::BrewUnit::brewTemp remains a settable, underived attribute for "
    "nominal (unchanged since Chapter 2); this record does not assume it has "
    "been derived from anything."
]
evidence_refs_s4 = [
    "Step 1's own requirement_coverage() output and tie-evidence search (this exercise, reproduced above)",
    "Step 2's own remediation and AC-C10-EX, cited above",
    "Step 3's own three reconstructed ReviewRecords, each independently validated (this exercise)",
]
for p in premises_s4:
    print(p)
print()
print(assumption_refs_s4)
print(evidence_refs_s4)

In [ ]:
signoff = ReviewRecord(
    identifier="AI-C10-EX",
    kind="asserted_inference",
    claim="# What this synthesis gives an accountable engineer: state plainly what is "
          "bidirectionally verified, what has only one-sided evidence, and that your "
          "own model's strongest formal proof is now tied to the new requirement your "
          "own Step 2 added (confirmed by Step 1's own two narrow, explicitly-named "
          "checks: satisfy-by-subject; a direct reference from within a requirement's "
          "own body) -- and that this record is not itself sign-off",
    model_ref="CoffeeDemo",
    content_hash=hash_content(remediated_source),
    scope="# The three chains this synthesis draws on: brewReq's own bidirectional coverage, tempCheck's own one-sided coverage, and deliveredMassBoundedBySupply's own tie, once missing and now closed by Step 2 (satisfy-by-subject; a direct subsets/redefines/references/referent reference from within a requirement's own body, exact-type owner only -- item 29, DL-070/DL-071/DL-072) -- confined to what Steps 1 through 3 actually established, not a claim about any part of the model none of them touched",
    criteria="# For each traced element: real bidirectional verification evidence, only one-sided evidence, or none at all? Does Step 1's own two narrow checks find a proven formal property connected back to a stated requirement? Here it now does, but only because Step 2 closed the gap Step 1 found, and AC-C10-EX already names the honest limits of what that closure actually establishes.",
    premises=premises_s4,
    assumption_refs=assumption_refs_s4,
    evidence_refs=evidence_refs_s4,
    rationale="# Which requirement is most strongly traced and why, which is weakest and why, how deliveredMassBoundedBySupply's own strength and the tie Step 2 added are two separate facts (one about the proof's own scope, one about when the requirement's need was identified -- mirror AC-C10-EX's own framing), and why no assert-satisfy claim was needed or used",
    counterevidence="# What is NOT shown: no physical realization of the coffee maker as a whole has been checked end-to-end; WaterMover's own mechanism selection rests on a domain premise, not a trade study; waterIn is not wired to a producer; BrewUnit::brewTemp is not derived from anything, so tempCheck's own coverage gap cannot be closed by more querying alone; and the tie Step 2 added rests on structural detection alone, with no independent solver-level re-verification (mirror AC-C10-EX's own counterevidence)",
    residual_uncertainties="# What is genuinely open: whether Impeller remains the right mechanism selection, whether the restated deliveredMassBoundedBySupply lemma would still hold if WaterMover's own construct were edited without updating the restatement, whether brewTemp should ever be derived and from what, and whether a requirement whose own need was identified after its own evidence should ever be accepted without qualification -- state plainly that none of this is decided here",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(signoff)
print(f"Validation errors: {errors}")
print(f"identifier={signoff.identifier!r}  kind={signoff.kind!r}")
print(f"disposition={signoff.disposition!r}  record_kind={signoff.record_kind!r}")
print(f"engineering_conclusion={signoff.engineering_conclusion!r}")
assert errors == []
assert signoff.disposition == "pending"
assert signoff.record_kind == "worked_example"

`engineering_conclusion` stays `undetermined`, not `supported`: one
requirement is genuinely covered, one is not, and although the model's own
strongest proof is now tied to the new requirement your own Step 2 added
(confirmed by Step 1's own two narrow, bounded checks, structurally via
Check B alone), `AC-C10-EX`'s own honest framing of that tie names a real,
unresolved limitation of its own -- so no single word honestly describes
the model as a whole except the word that admits the mixture -- the same
reading real Chapter 10's own synthesis gives the toaster. Say plainly what
this record is, and is not: it is a synthesis of two real, already-verified
inputs into one honest, bounded statement of what is established, what is
not, and what residual judgment remains. It is not sign-off. A completed
traceability graph and judgment ledger tell an engineer what they are
working with; deciding whether that is enough to actually proceed is a
human, accountable act this exercise can show the inputs to but cannot
perform on your behalf.

Four steps, one arc: a real traceability graph that traces two
requirements very differently and finds the model's own strongest proof
tied to no requirement at all; a direct remediation that closes that gap
structurally, by subsetting, not by `assert satisfy`, with `AC-C10-EX`
honestly framing what the tie does and does not establish; a judgment
ledger over three of your own real records; and a synthesis record that
states honestly what all of that does, and does not, establish, with
`disposition="pending"` like every other record this tutorial has built.

AGENTS.md's own account of emergence names what a real sign-off actually
has to weigh, beyond anything this record states: "strong emergence
(unanticipated; seen only in integration, test or operation) belongs to no
layer. It is what sign-off judges." This exercise's own synthesis, however
honestly built and however carefully bounded, cannot anticipate strong
emergence by construction: it is a map of what has already been checked
and what has not, not a forecast of what integration or operation might
still reveal. It is exactly the kind of artifact a real engineer would
weigh against that residual, unanticipated risk before deciding to
proceed, not a substitute for weighing it. A traceable, honestly-scoped
case makes that judgment easier to exercise well; it does not make the
judgment itself unnecessary.

In [ ]:
conn.close()